# HW1 布尔查询之BSBI与索引压缩

本次作业使用斯坦福大学[CS 276 / LING 286: Information Retrieval and Web Search](https://web.stanford.edu/class/cs276/)课程的代码框架来实现。具体来说主要包含的内容有：
1. [索引构建 (40%)](#索引构建与检索-(40%)) 使用BSBI方法模拟在内存不足的情况下的索引构建方式，并应用于布尔查询
2. [索引压缩 (30%)](#索引压缩-(30%)) 使用可变长编码对构建的索引进行压缩
3. [布尔检索 (10%)](#布尔联合检索-(10%)) 对空格分隔的单词查询进行联合（与）布尔检索
3. [实验报告 (10%)](#Report-(25%)) 描述你的代码并回答一些问题
4. [额外的编码方式 (10%)](#额外的编码方式-(10%)) 鼓励使用额外的编码方式对索引进行压缩 (例如, gamma-encoding)

<div style="background:#f7fafc; border-left:4px solid #4299e1; padding:10px 14px; border-radius:4px;">
<b>写在前面：</b> <br>
在本次实验中，由于代码量较大，实验报告部分将以Markdown块的形式完成；为了与原有文本进行区分，报告内容均使用侧边高亮文本框进行呈现。<br>
</div>

<div style="background:#f7fafc; border-left:4px solid #4299e1; padding:10px 14px; border-radius:4px;">
<b>代码仓库链接：</b>
Github：<a href="https://github.com/Iriasoul/Information-Retrieval" target="_blank">https://github.com/Iriasoul/Information-Retrieval</a>
</div>


In [1]:
# You can add additional imports here
import sys
import pickle as pkl
import array
import os
import timeit
import contextlib

# 数据集

实验使用的文本数据是stanford.edu域下的网页内容，可从http://web.stanford.edu/class/cs276/pa/pa1-data.zip 下载。以下代码将大约170MB的文本数据下载到当前目录下，

In [2]:
import urllib.request
import zipfile

# data_url = 'http://web.stanford.edu/class/cs276/pa/pa1-data.zip'
# data_dir = 'pa1-data'
# urllib.request.urlretrieve(data_url, data_dir+'.zip')
# zip_ref = zipfile.ZipFile(data_dir+'.zip', 'r')
# zip_ref.extractall()
# zip_ref.close()

<div style="background:#f7fafc; border-left:4px solid #4299e1; padding:10px 14px; border-radius:4px;">
注：此处下载的文件已经在实验文件中给出，由于文件内容过多，实验中采用手动下载完成
</div>    

之后构建的索引会被存储到`output_dir`，`tmp`会存储测试数据（toy-data）所生成的一些临时文件

In [3]:
try: 
    os.mkdir('output_dir')
except FileExistsError:
    pass
try: 
    os.mkdir('tmp')
except FileExistsError:
    pass
try: 
    os.mkdir('toy_output_dir')
except FileExistsError:
    pass

在数据目录下有10个子目录（命名0-9）

In [4]:
sorted(os.listdir('pa1-data/pa1-data'))

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']

每一个子目录下的文件都包含一个独立网页的内容。可以认为在同一子目录下没有同名文件，即每个文件的绝对路径不会相同。

In [5]:
sorted(os.listdir('pa1-data/pa1-data/0'))[:10]

['3dradiology.stanford.edu_',
 '3dradiology.stanford.edu_patient_care_Case%2520studies_AVM.html',
 '3dradiology.stanford.edu_patient_care_case_studies.html',
 '5-sure.stanford.edu_',
 '50years.stanford.edu_',
 'a3cservices.stanford.edu_awards_nominate_',
 'a3cservices.stanford.edu_facilities_',
 'a3cservices.stanford.edu_lead_',
 'aa.stanford.edu_',
 'aa.stanford.edu_about_aviation.php']

所有的网页内容已经经过处理，仅包含由空格分隔开的单词，不再需要进行额外的标准化工作。

In [6]:
with open('pa1-data/pa1-data/0/3dradiology.stanford.edu_', 'r') as f:
    print(f.read())

3d radiology lab stanford university school of medicine stanford school of medicine 3d and quantitative imaging in the department of radiology search this site only stanford medical sites ways to give find a person alumni lane library ways to give find a person about us mission to develop and apply innovative techniques for efficient quantitative analysis and display of medical imaging data through interdisciplinary collaboration goals education to train physicians and technologists locally and worldwide in the latest developments in 3d and quantitative imaging research to develop new approaches to the exploration analysis and quantitative assesment of diagnostic images that result in a new and or more cost effective diagnostic approaches and b new techniques for the design and planning and monitoring of therapy patient care to deliver valid clinically relevant visualization and analysis of medical imaging data to the stanford community locations richard m lucas magnetic resonance imag

    这里好像缺了一行，后面报错 toy_dir 未定义，此处补上文件路径

In [7]:
toy_dir = 'toy-data/toy-data/'

# 索引构建与检索 (40%)

作业的第一部分是使用**blocked sort-based indexing (BSBI)** 算法来构建倒排索引并实现布尔检索。关于BSBI算法可以参考老师课件或者斯坦福教材[Section 4.2](http://nlp.stanford.edu/IR-book/pdf/04const.pdf)。以下摘自教材内容

> To construct an index, we first make a pass through the collection assembling all term-docID pairs. We then sort the pairs with the term as the dominant key and docID as the secondary key. Finally, we organize the docIDs for each term into a postings list and compute statistics like term and document frequency. For small collections, all this can be done in memory. 

对于无法在内存一次性处理的较大数据集，将会使用到二级存储（如：磁盘）。

## IdMap (6%)

再次引用教材 Section 4.2:

> To make index construction more efficient, we represent terms as termIDs (instead of strings), where each termID is a unique serial number. We can build the mapping from terms to termIDs on the fly while we are processing the collection. Similarly, we also represent documents as docIDs (instead of strings).

我们首先定义一个辅助类`IdMap`，用于将字符串和数字ID进行相互映射，以满足我们在term和termID、doc和docID间转换的需求。

实现以下代码中的`_get_str` 和 `_get_id`函数，IdMap类的唯一接口是`__getitem__`，它是一个特殊函数，重写了下标运算`[]`,根据下标运算键的类型得到正确的映射值（如果不存在需要添加）。（特殊函数可参考[官方文档](https://docs.python.org/3.7/reference/datamodel.html#special-method-names)）
<br>
<br>
我们会用到字典来将字符串转换为数字，用列表来将数字转换为字符串。(4%)

<div style="background:#f7fafc; border-left:4px solid #4299e1; padding:10px 14px; border-radius:4px;">
为了构建索引，我们首先需要实现一个可靠的映射。在这一部分我们将完成IdMap类的_get_str和_get_id功能，分别使用列表与字典实现两者的查询，以便后续操作。<br><br>
在实际使用中，每次遇见一个新的字符串，就会为它分配一个新的ID作为Value，因此，_get_id应该具备新建映射的功能。具体实现上，要先判断查询的字符串Key是否已经存在于字典中：如果不是，则将<字符串，ID>键值对存入字典， 同时将该字符串存入列表中；如果是，就直接利用字典的特性得到相应的Value，即字符串的ID 。对于_get_str，则只需使用ID下标访问列表，获取相应元素即可。<br><br>在ID的选取上，此处使用字典的长度实现。每次新建映射时，字典长度一定不会重复，同时又因为字典和列表的长度也一定相同，当ID作为下标时从列表中取出字符串也一定是对应的。
</div>

In [8]:
class IdMap:
    """Helper class to store a mapping from strings to ids."""
    def __init__(self):
        self.str_to_id = {}
        self.id_to_str = []
        
    def __len__(self):
        """Return number of terms stored in the IdMap"""
        return len(self.id_to_str)
        
    def _get_str(self, i):
        """Returns the string corresponding to a given id (`i`)."""
        ### Begin your code
        return self.id_to_str[i]
        ### End your code
        
    def _get_id(self, s):
        """Returns the id corresponding to a string (`s`). 
        If `s` is not in the IdMap yet, then assigns a new id and returns the new id.
        """
        ### Begin your code
        if s not in self.str_to_id:
            # 新ID = 当前列表长度
            new_id = len(self.id_to_str)
            self.str_to_id[s] = new_id
            self.id_to_str.append(s)
        return self.str_to_id[s]
        ### End your code
            
    def __getitem__(self, key):
        """If `key` is a integer, use _get_str; 
           If `key` is a string, use _get_id;"""
        if type(key) is int:
            return self._get_str(key)
        elif type(key) is str:
            return self._get_id(key)
        else:
            raise TypeError

确保代码能通过以下简单测试样例 (2%)

In [9]:
testIdMap = IdMap()
assert testIdMap['a'] == 0, "Unable to add a new string to the IdMap"
assert testIdMap['bcd'] == 1, "Unable to add a new string to the IdMap"
assert testIdMap['a'] == 0, "Unable to retrieve the id of an existing string"
assert testIdMap[1] == 'bcd', "Unable to retrive the string corresponding to a\
                                given id"
try:
    testIdMap[2]
except IndexError as e:
    assert True, "Doesn't throw an IndexError for out of range numeric ids"
assert len(testIdMap) == 2

之后会需要你自己来写测试样例来确保你的程序正常运行

## 将倒排列表编码成字节数组 (2%)

为了高效地从磁盘读写倒排列表（文档ID），我们将其存储为字节数组的形式。代码提供了`UncompressedPostings`类来用静态函数实现对倒排列表的编码和解码。在之后的任务中你需要使用该接口实现索引压缩版本（可变长编码）。

参考:
1. https://docs.python.org/3/library/array.html
2. https://pymotw.com/3/array/#module-array

In [10]:
class UncompressedPostings:
    
    @staticmethod
    def encode(postings_list):
        """Encodes postings_list into a stream of bytes
        
        Parameters
        ----------
        postings_list: List[int]
            List of docIDs (postings)
            
        Returns
        -------
        bytes
            bytearray representing integers in the postings_list
        """
        return array.array('L', postings_list).tobytes()
        
    @staticmethod
    def decode(encoded_postings_list):
        """Decodes postings_list from a stream of bytes
        
        Parameters
        ----------
        encoded_postings_list: bytes
            bytearray representing encoded postings list as output by encode 
            function
            
        Returns
        -------
        List[int]
            Decoded list of docIDs from encoded_postings_list
        """
        
        decoded_postings_list = array.array('L')
        decoded_postings_list.frombytes(encoded_postings_list)
        return decoded_postings_list.tolist()

运行以下代码查看其工作方式 (2%)

In [11]:
x = UncompressedPostings.encode([1,2,3])
print(x)
print(UncompressedPostings.decode(x))

b'\x01\x00\x00\x00\x02\x00\x00\x00\x03\x00\x00\x00'
[1, 2, 3]


## 磁盘上的倒排索引 (2%)

> With main memory insufficient, we need to use an external sorting algorithm, that is, one that uses disk. For acceptable speed, the central requirement of such an algorithm is that it minimize the number of random disk seeks during sorting - sequential disk reads are far faster than seeks. 

在这一部分我们提供了一个基类`InvertedIndex`，之后会在此基础上构建它的子类`InvertedIndexWriter`, `InvertedIndexIterator` 和 `InvertedIndexMapper`。在Python中我们常用`cPickle`进行序列化，但是它并不支持部分读和部分写，无法满足BSBI算法的需要，所以我们需要定义自己的存储方式。

In [12]:
class InvertedIndex:
    """A class that implements efficient reads and writes of an inverted index 
    to disk
    
    Attributes
    ----------
    postings_dict: Dictionary mapping: termID->(start_position_in_index_file, 
                                                number_of_postings_in_list,
                                               length_in_bytes_of_postings_list)
        This is a dictionary that maps from termIDs to a 3-tuple of metadata 
        that is helpful in reading and writing the postings in the index file 
        to/from disk. This mapping is supposed to be kept in memory. 
        start_position_in_index_file is the position (in bytes) of the postings 
        list in the index file
        number_of_postings_in_list is the number of postings (docIDs) in the 
        postings list
        length_in_bytes_of_postings_list is the length of the byte 
        encoding of the postings list
    
    terms: List[int]
        A list of termIDs to remember the order in which terms and their 
        postings lists were added to index. 
        
        After Python 3.7 we technically no longer need it because a Python dict 
        is an OrderedDict, but since it is a relatively new feature, we still
        maintain backward compatibility with a list to keep track of order of 
        insertion. 
    """
    def __init__(self, index_name, postings_encoding=None, directory=''):
        """
        Parameters
        ----------
        index_name (str): Name used to store files related to the index
        postings_encoding: A class implementing static methods for encoding and 
            decoding lists of integers. Default is None, which gets replaced
            with UncompressedPostings
        directory (str): Directory where the index files will be stored
        """

        self.index_file_path = os.path.join(directory, index_name+'.index')
        self.metadata_file_path = os.path.join(directory, index_name+'.dict')

        if postings_encoding is None:
            self.postings_encoding = UncompressedPostings
        else:
            self.postings_encoding = postings_encoding
        self.directory = directory

        self.postings_dict = {}
        self.terms = []         #Need to keep track of the order in which the 
                                #terms were inserted. Would be unnecessary 
                                #from Python 3.7 onwards

    def __enter__(self):
        """Opens the index_file and loads metadata upon entering the context"""
        # Open the index file
        self.index_file = open(self.index_file_path, 'rb+')

        # Load the postings dict and terms from the metadata file
        with open(self.metadata_file_path, 'rb') as f:
            self.postings_dict, self.terms = pkl.load(f)
            self.term_iter = self.terms.__iter__()                       

        return self
    
    def __exit__(self, exception_type, exception_value, traceback):
        """Closes the index_file and saves metadata upon exiting the context"""
        # Close the index file
        self.index_file.close()
        
        # Write the postings dict and terms to the metadata file
        with open(self.metadata_file_path, 'wb') as f:
            pkl.dump([self.postings_dict, self.terms], f)

因为是在与磁盘上的文件进行交互，所以我们提供了`__enter__`和`__exit__`函数，它使得我们能够像使用python中文件IO一样使用`with`语句。（参考[上下文管理器官方文档](https://docs.python.org/3/library/contextlib.html)）

以下是使用 `InvertedIndexWriter` 上下文管理器的实例: (2%)

```python
with InvertedIndexWriter('test', directory='tmp/') as index:
    # Some code here
```

## 索引 (30%)

> BSBI (i) segments the collection into parts of equal size, (ii) sorts the termID-docID pairs of each part in memory, (iii) stores intermediate sorted results on disk, and (iv) merges all intermediate results into the final index. 

你需要将每一个子目录当做一个块（block），并且在构建索引的过程中每次只能加载一个块（模拟内存不足）。注意到我们是将操作系统意义上的块进行了抽象。你可以认为每个块足够小，能被装载进内存。

在这一部分，我们将阶段性地构造类`BSBIIndex`。函数`index`给出了BSBI算法的框架，而你的工作则是在接下来的部分中实现函数`parse_block`, `invert_write` 和 `merge`。

In [13]:
# Do not make any changes here, they will be overwritten while grading
class BSBIIndex:
    """ 
    Attributes
    ----------
    term_id_map(IdMap): For mapping terms to termIDs
    doc_id_map(IdMap): For mapping relative paths of documents (eg 
        0/3dradiology.stanford.edu_) to docIDs
    data_dir(str): Path to data
    output_dir(str): Path to output index files
    index_name(str): Name assigned to index
    postings_encoding: Encoding used for storing the postings.
        The default (None) implies UncompressedPostings
    """
    def __init__(self, data_dir, output_dir, index_name = "BSBI", 
                 postings_encoding = None):
        self.term_id_map = IdMap()
        self.doc_id_map = IdMap()
        self.data_dir = data_dir
        self.output_dir = output_dir
        self.index_name = index_name
        self.postings_encoding = postings_encoding

        # Stores names of intermediate indices
        self.intermediate_indices = []
        
    def save(self):
        """Dumps doc_id_map and term_id_map into output directory"""
        
        with open(os.path.join(self.output_dir, 'terms.dict'), 'wb') as f:
            pkl.dump(self.term_id_map, f)
        with open(os.path.join(self.output_dir, 'docs.dict'), 'wb') as f:
            pkl.dump(self.doc_id_map, f)
    
    def load(self):
        """Loads doc_id_map and term_id_map from output directory"""
        
        with open(os.path.join(self.output_dir, 'terms.dict'), 'rb') as f:
            self.term_id_map = pkl.load(f)
        with open(os.path.join(self.output_dir, 'docs.dict'), 'rb') as f:
            self.doc_id_map = pkl.load(f)
            
    def index(self):
        """Base indexing code
        
        This function loops through the data directories, 
        calls parse_block to parse the documents
        calls invert_write, which inverts each block and writes to a new index
        then saves the id maps and calls merge on the intermediate indices
        """
        for block_dir_relative in sorted(next(os.walk(self.data_dir))[1]):
            td_pairs = self.parse_block(block_dir_relative)
            index_id = 'index_'+block_dir_relative
            self.intermediate_indices.append(index_id)
            with InvertedIndexWriter(index_id, directory=self.output_dir, 
                                     postings_encoding=
                                     self.postings_encoding) as index:
                self.invert_write(td_pairs, index)
                td_pairs = None
        self.save()
        with InvertedIndexWriter(self.index_name, directory=self.output_dir, 
                                 postings_encoding=
                                 self.postings_encoding) as merged_index:
            with contextlib.ExitStack() as stack:
                indices = [stack.enter_context(
                    InvertedIndexIterator(index_id, 
                                          directory=self.output_dir, 
                                          postings_encoding=
                                          self.postings_encoding)) 
                 for index_id in self.intermediate_indices]
                self.merge(indices, merged_index)

### 解析 (10%)

> The function `parse_block`  parses documents into termID-docID pairs and accumulates the pairs in memory until a block of a fixed size is full. We choose the block size to fit comfortably into memory to permit a fast in-memory sort. 

你需要将每一个子目录当做一个块，`parse_block`接收子目录路径作为参数。同一子目录下所有文件名都是不同的。 (5%)

_注意 - 我们使用 `BSBIIndex` 继承 `BSBIIndex`, 这只是对一个已存在类添加新内容的简单方法。在这里只是用来切分类的定义（jupyter notebook内教学使用，无特殊含义）。_

<div style="background:#f7fafc; border-left:4px solid #4299e1; padding:10px 14px; border-radius:4px;">
在完成了IdMap与倒排表编码这些基础工作后，将进入BSBI的核心部分。观察实验给出的文件集，可以看到它们被分为了10个文件夹（标号0~9），BSBI的核心流程就是在每个分块内解析并排序，生成以块为单位倒排表，再将它们合并最终得到全局的倒排索引。<br><br>

在这里，我们首先需要完成parse_block这一方法，它的主要功能是将文件中的单词流转换为（term_id,doc_id），再将得到的所有元组存入列表作为返回值。具体实现方面，拼接好文件地址后，遍历读取文件，为文件名分配doc_id，文件里的内容则是拆成一个个单词为其分配term_id，最后将(term_id, doc_id)组合成元组存到列表里即可。
</div>

In [14]:
class BSBIIndex(BSBIIndex):            
    def parse_block(self, block_dir_relative):
        """Parses a tokenized text file into termID-docID pairs
        
        Parameters
        ----------
        block_dir_relative : str
            Relative Path to the directory that contains the files for the block
        
        Returns
        -------
        List[Tuple[Int, Int]]
            Returns all the td_pairs extracted from the block
        
        Should use self.term_id_map and self.doc_id_map to get termIDs and docIDs.
        These persist across calls to parse_block
        """
        ### Begin your code
        
        result_list = []
        root_addr = os.path.join(self.data_dir, block_dir_relative)
        addr_list = os.listdir(root_addr)
        for addr in sorted(addr_list):
            with open(os.path.join(root_addr, addr), 'r') as f:
                doc_id = self.doc_id_map._get_id(os.path.join(block_dir_relative, addr))
                word_list = f.read().strip().split()
                for word in word_list:
                    term_id = self.term_id_map._get_id(word)
                    term_doc_pair = [term_id, doc_id]
                    result_list.append(term_doc_pair)
        return result_list
    
        ### End your code

观察函数在测试数据上是否正常运行 (2%)

In [15]:
with open('toy-data/toy-data/0/fine.txt', 'r') as f:
    print(f.read())
with open('toy-data/toy-data/0/hello.txt', 'r') as f:
    print(f.read())

i'm fine , thank you

hi hi
how are you ?



In [16]:
BSBI_instance = BSBIIndex(data_dir=toy_dir, output_dir = 'tmp/', index_name = 'toy')
BSBI_instance.parse_block('0')

[[0, 0],
 [1, 0],
 [2, 0],
 [3, 0],
 [4, 0],
 [5, 1],
 [5, 1],
 [6, 1],
 [7, 1],
 [4, 1],
 [8, 1]]

写一些测试样例来确保`parse_block`方法正常运行（如：相同单词出现时是相同ID） (3%)

In [17]:
### Begin your code
import tempfile
import shutil

# 创建临时目录
temp_dir = tempfile.mkdtemp()
try:
    # 创建测试用的块目录
    block_dir = os.path.join(temp_dir, "test_block")
    os.makedirs(block_dir, exist_ok=True)

    # 测试文档
    with open(os.path.join(block_dir, "doc1.txt"), "w") as f:
        f.write("apple apple orange")  # doc1：apple出现2次
    with open(os.path.join(block_dir, "doc2.txt"), "w") as f:
        f.write("banana apple")       # doc2：apple出现1次

    # 初始化
    bsbii = BSBIIndex(data_dir=temp_dir, output_dir='tmp/', index_name='test')
    td_pairs = bsbii.parse_block(block_dir)  # 解析刚才创建的块

    # 打印结果
    print("解析结果 (termID, docID)：")
    for pair in td_pairs:
        print(pair)
    apple_term_id = bsbii.term_id_map["apple"]
    print(f"\n验证：单词'apple'的唯一termID = {apple_term_id}")
    # 遍历所有对，找出apple对应的记录
    for term_id, doc_id in td_pairs:
        if term_id == apple_term_id:
            print(f"→ apple出现在 docID={doc_id} 中，termID={term_id}（一致）")

finally:
    # 删除临时目录
    shutil.rmtree(temp_dir)
### End your code

解析结果 (termID, docID)：
[0, 0]
[0, 0]
[1, 0]
[2, 1]
[0, 1]

验证：单词'apple'的唯一termID = 0
→ apple出现在 docID=0 中，termID=0（一致）
→ apple出现在 docID=0 中，termID=0（一致）
→ apple出现在 docID=1 中，termID=0（一致）


### 倒排表 (10%)

> The block is then inverted and written to disk. Inversion involves two steps. First, we sort the termID-docID pairs. Next, we collect all termID-docID pairs with the same termID into a postings list, where a posting is simply a docID. The result, an inverted index for the block we have just read, is then written to disk.

在这一部分我们添加函数`invert_write`来实现由termID-docID对构建倒排表。 

但是，我们首先需要实现类`InvertedIndexWriter`。和列表类似，该类提供了append函数，但是倒排表不会存储在内存中而是直接写入到磁盘里。 (3%)

<div style="background:#f7fafc; border-left:4px solid #4299e1; padding:10px 14px; border-radius:4px;">
本部分我们继续进行倒排表的构建。这里要实现InvertedIndexWriter类的append方法，具体要完成的内容如下：首先对传入的倒排列表进行编码，获取编码后的字节流及其长度；随后以追加模式打开索引文件进行编辑，若当前词项未被记录，则将其加入词项列表，并在字典中存储文件偏移量、列表长度和编码长度作为元数据，最后将编码后的字节流写入文件并关闭文件，完成索引的追加存储。
</div>

In [18]:
class InvertedIndexWriter(InvertedIndex):
    """"""
    def __enter__(self):
        os.makedirs(os.path.dirname(self.index_file_path), exist_ok=True)
        self.index_file = open(self.index_file_path, 'wb+')              
        return self

    def append(self, term, postings_list):
        """Appends the term and postings_list to end of the index file.
        
        This function does three things, 
        1. Encodes the postings_list using self.postings_encoding
        2. Stores metadata in the form of self.terms and self.postings_dict
           Note that self.postings_dict maps termID to a 3 tuple of 
           (start_position_in_index_file, 
           number_of_postings_in_list, 
           length_in_bytes_of_postings_list)
        3. Appends the bytestream to the index file on disk

        Hint: You might find it helpful to read the Python I/O docs
        (https://docs.python.org/3/tutorial/inputoutput.html) for
        information about appending to the end of a file.
        
        Parameters
        ----------
        term:
            term or termID is the unique identifier for the term
        postings_list: List[Int]
            List of docIDs where the term appears
        """
        ### Begin your code
        ep_list=self.postings_encoding.encode(postings_list)
        encode_len=len(ep_list)
        with open(self.index_file_path,'ab') as index_input:
            if self.postings_dict.get(term) is None:
                self.terms.append(term)
                self.postings_dict[term] = (index_input.tell(), len(postings_list), encode_len)
            index_input.write(ep_list)
            index_input.close()
        ### End your code

尽管还没有实现读取索引的类，我们还是可以用以下测试代码检测我们的实现。 (2%)

In [19]:
with InvertedIndexWriter('test', directory='tmp/') as index:
    index.append(1, [2, 3, 4])
    index.append(2, [3, 4, 5])
    index.index_file.seek(0)
    assert index.terms == [1,2], "terms sequence incorrect"
    assert index.postings_dict == {1: (0, 3, len(UncompressedPostings.encode([2,3,4]))), 
                                   2: (len(UncompressedPostings.encode([2,3,4])), 3, 
                                       len(UncompressedPostings.encode([3,4,5])))}, "postings_dict incorrect"
    assert UncompressedPostings.decode(index.index_file.read()) == [2, 3, 4, 3, 4, 5], "postings on disk incorrect"

现在我们实现 `invert_write`，它将解析得到的td_pairs转换成倒排表，并使用`InvertedIndexWriter` 类将其写入磁盘。 (3%)

<div style="background:#f7fafc; border-left:4px solid #4299e1; padding:10px 14px; border-radius:4px;">
接下来实现invert_write方法，将前面得到的列表转换为倒排表，并用append方法写入磁盘。<br><br>
以(term,Postings_list)的形式构造字典，遍历参数列表，如果元组的term没有在字典里，则分配空列表；如果已经存在，则将doc_id取出放入相应的列表。然后将字典按照term_id进行排序，使用 InvertedIndexWriter的append方法，将term_id和其对应的已经排好序的postings_list写入索引文件。
</div>

In [20]:
class BSBIIndex(BSBIIndex):
    def invert_write(self, td_pairs, index):
        """Inverts td_pairs into postings_lists and writes them to the given index
        
        Parameters
        ----------
        td_pairs: List[Tuple[Int, Int]]
            List of termID-docID pairs
        index: InvertedIndexWriter
            Inverted index on disk corresponding to the block       
        """
        ### Begin your code
        if not td_pairs:
            return
        
        result_dict = {}
        for t,d in td_pairs:
            if result_dict.get(t) is None:
                result_dict[t] = []
            result_dict[t].append(d)
        seq_key = sorted(result_dict.keys())
        for i in seq_key:
            result_dict[i] = sorted(result_dict[i])
            index.append(i, result_dict[i])
        ### End your code

我们可以在测试数据上读取一个块并观察倒排索引中包含的内容。
仿照`InvertedIndexWriter`部分写一些测试样例。 (2%)

In [21]:
### Begin your code
# BSBI初始化
bsbi_instance = BSBIIndex(data_dir='toy-data/toy-data/', output_dir='tmp/', index_name='test_invert')

# 构造测试用的排序后 td_pairs
test_td_pairs = [(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (2, 5)]

with InvertedIndexWriter(
    index_name='test_invert',
    directory='tmp/',
    postings_encoding=UncompressedPostings
) as index_writer:
    bsbi_instance.invert_write(test_td_pairs, index_writer)
    
    # 验证 1：term 写入顺序与输入一致
    assert index_writer.terms == [1, 2], "term 顺序错误"
    
    # 验证 2：postings_dict 元数据正确
    postings_1_len = len(UncompressedPostings.encode([2, 3, 4]))
    assert index_writer.postings_dict[1] == (0, 3, postings_1_len), "term 1 的 postings 元数据错误"
    
    postings_2_len = len(UncompressedPostings.encode([3, 4, 5]))
    assert index_writer.postings_dict[2] == (postings_1_len, 3, postings_2_len), "term 2 的 postings 元数据错误"
    
    # 验证 3：磁盘数据解码后与原始 postings 一致
    index_writer.index_file.seek(0)
    decoded_postings = UncompressedPostings.decode(index_writer.index_file.read())
    assert decoded_postings == [2, 3, 4, 3, 4, 5], "磁盘 postings 解码错误"

print("✅ invert_write 所有测试通过！")
### End your code

✅ invert_write 所有测试通过！


### 合并 (10%)
> The algorithm simultaneously merges the ten blocks into one large merged index. To do the merging, we open all block files simultaneously, and maintain small read buffers for the ten blocks we are reading and a write buffer for the final merged index we are writing. 

Python中的迭代模型非常自然地符合我们维护一个小的读缓存的要求。我们可以迭代地从磁盘上每次读取文件的一个倒排列表。我们通过构建`InvertedIndex`的子类`InvertedIndexIterator`来完成这个迭代任务。 (3%)

<div style="background:#f7fafc; border-left:4px solid #4299e1; padding:10px 14px; border-radius:4px;">
通过之前的步骤，已经实现了每个文件块的索引构建，在本部分，将完成合并操作，最终得到全局的倒排索引。<br><br>
在实现块索引的合并之前，首先要完成一个用于访问块索引的迭代器，也就是此处的InvertedIndexIterator类，主要需要实现_initialization_hook和__next__这两个方法。前者只需进行简单的初始化即可；后者则需维护指针的迭代，并实现相应的操作。<br><br>
对于__next__方法，需要先判断当前指针是否已经越界，如果越界则直接抛出StopIteration终止迭代。如果不越界则继续进行操作：先从terms列表中取出对应的term，再从postings_dict字典中获取该term对应的起始位置、词条数量与字节长度的三元组信息，接着根据起始位置将索引文件指针定位到对应位置，这里仅读取指定字节长度的编码数据，减少内存加载；然后对读取到的编码数据进行解码得到postings_list，再将迭代指针向后移动。最后把获取到的term和对应的postings_list打包成元组返回即可。
</div>

In [22]:
class InvertedIndexIterator(InvertedIndex):
    """"""
    def __enter__(self):
        """Adds an initialization_hook to the __enter__ function of super class
        """
        super().__enter__()
        self._initialization_hook()
        return self

    def _initialization_hook(self):
        """Use this function to initialize the iterator
        """
        ### Begin your code
        self.current_term_idx = 0
        self.index_file.seek(0) # 重置文件指针
        ### End your code

    def __iter__(self): 
        return self
    
    def __next__(self):
        """Returns the next (term, postings_list) pair in the index.
        
        Note: This function should only read a small amount of data from the 
        index file. In particular, you should not try to maintain the full 
        index file in memory.
        """
        ### Begin your code 
        if self.current_term_idx >= len(self.terms):
            raise StopIteration
        
        current_term = self.terms[self.current_term_idx]
        start_pos, num_postings, byte_length = self.postings_dict[current_term]
        
        self.index_file.seek(start_pos)

        # 仅读取当前 term 对应的字节数据（避免加载整个索引）
        encoded_postings = self.index_file.read(byte_length)
        postings_list = self.postings_encoding.decode(encoded_postings)
        
        self.current_term_idx += 1
        
        return (current_term, postings_list)
        ### End your code

    def delete_from_disk(self):
        """Marks the index for deletion upon exit. Useful for temporary indices
        """
        self.delete_upon_exit = True

    def __exit__(self, exception_type, exception_value, traceback):
        """Delete the index file upon exiting the context along with the
        functions of the super class __exit__ function"""
        self.index_file.close()
        if hasattr(self, 'delete_upon_exit') and self.delete_upon_exit:
            os.remove(self.index_file_path)
            os.remove(self.metadata_file_path)
        else:
            with open(self.metadata_file_path, 'wb') as f:
                pkl.dump([self.postings_dict, self.terms], f)

为了测试以上代码，我们先用`InvertedIndexWriter` 创建索引，然后再迭代遍历。写一些小的测试样例观察它是否正常运行。至少你得写一个测试，手工构建一个小的索引，用`InvertedIndexWriter`将他们写入磁盘，然后用`InvertedIndexIterator`迭代遍历倒排索引。 (2%)

In [23]:
### Begin your code
# 创建测试索引
with InvertedIndexWriter('test_iter', directory='tmp/', postings_encoding=UncompressedPostings) as writer:
    writer.append(1, [2, 3, 4])
    writer.append(2, [3, 4, 5])
    writer.append(3, [5, 6])

# 迭代器遍历测试
with InvertedIndexIterator('test_iter', directory='tmp/', postings_encoding=UncompressedPostings) as iterator:
    print("迭代器遍历结果：")
    for term, postings in iterator:
        print(f"term: {term} → postings: {postings}")

### End your code

迭代器遍历结果：
term: 1 → postings: [2, 3, 4]
term: 2 → postings: [3, 4, 5]
term: 3 → postings: [5, 6]


> During merging, in each iteration, we select the lowest termID that has not been processed yet using a priority queue or a similar data structure. All postings lists for this termID are read and merged, and the merged list is written back to disk. Each read buffer is refilled from its file when necessary.

我们将使用`InvertedIndexIterator`来完成读取的部分，用`InvertedIndexWriter`来将合并后的索引写入磁盘。

注意到我们之前一直使用`with` 语句来打开倒排索引文件，但是现在需要程序化地完成这项工作。可以看到我们在基类`BSBIIndex`的`index`函数中使用了`contextlib`([参考文档](https://docs.python.org/3/library/contextlib.html#contextlib.ExitStack))
你的任务是合并*打开的*`InvertedIndexIterator`对象（倒排列表），并且通过一个`InvertedIndexWriter`对象每次写入一个文档列表。

既然我们知道文档列表已经排过序了，那么我们可以在线性时间内对它们进行合并排序。事实上`heapq`([参考文档](https://docs.python.org/3.0/library/heapq.html)) 是Python中一个实现了堆排序算法的标准模板。另外它还包含一个实用的函数`heapq.merge`，可以将多个已排好序的输入（可迭代）合并成一个有序的输出并返回它的迭代器。它不仅可以用来合并倒排列表，也可以合并倒排索引词典。

为了让你快速上手`heapq.merge`函数，我们提供了一个简单的使用样例。给出两个以动物和树平均寿命排序的列表，我们希望合并它们。

In [24]:
import heapq
animal_lifespans = [('Giraffe', 28), 
                   ('Rhinoceros', 40), 
                   ('Indian Elephant', 70), 
                   ('Golden Eagle', 80), 
                   ('Box turtle', 123)]

tree_lifespans = [('Gray Birch', 50), 
                  ('Black Willow', 70), 
                  ('Basswood', 100),
                  ('Bald Cypress', 600)]

lifespan_lists = [animal_lifespans, tree_lifespans]

for merged_item in heapq.merge(*lifespan_lists, key=lambda x: x[1]):
    print(merged_item)

('Giraffe', 28)
('Rhinoceros', 40)
('Gray Birch', 50)
('Indian Elephant', 70)
('Black Willow', 70)
('Golden Eagle', 80)
('Basswood', 100)
('Box turtle', 123)
('Bald Cypress', 600)


注意使用`*`将`lifespan_lists`解包作为参数，并且使用`lambda`函数来给出进行排序的key。如果你对它们不熟悉可以参考文档([unpacking lists](https://docs.python.org/3/tutorial/controlflow.html#unpacking-argument-lists), [lambda expressions](https://docs.python.org/3/tutorial/controlflow.html#lambda-expressions))。 (3%)

<div style="background:#f7fafc; border-left:4px solid #4299e1; padding:10px 14px; border-radius:4px;">
对于merge方法，这里使用temp_term和temp_list记录上一个term及其对应的合并后的倒排表，在循环遍历过程中通过这两个临时变量来对比相邻的term和postings_list，从而完成倒排表的合并。具体操作上，先使用heapq.merge对多个索引迭代器进行处理，让所有(term, postings_list)元组都按照term从小到大有序排列，之后开始对处理后的有序元素进行遍历。<br><br>
在遍历过程中，如果当前term与临时记录的temp_term不相等，会先判断temp_term是否为空，若是则代表当前是第一个元素，直接将当前的 term 和 postings_list 赋值给临时变量即可，若不是则说明遇到了新的不同term，此时先将临时记录的term和去重排序后的倒排列表写入合并索引（使用append方法），再把临时变量更新为当前的term和postings_list；如果当前term与temp_term相等，就直接将当前的postings_list拼接到临时记录的倒排列表中。整个遍历循环结束后，最后一组临时记录的term和合并后的倒排列表不会在循环中被写入，因此需要单独判断并将其写入合并索引，最终完成所有索引的合并操作。
</div>

In [25]:
import heapq
class BSBIIndex(BSBIIndex):
    def merge(self, indices, merged_index):
        """Merges multiple inverted indices into a single index
        
        Parameters
        ----------
        indices: List[InvertedIndexIterator]
            A list of InvertedIndexIterator objects, each representing an
            iterable inverted index for a block
        merged_index: InvertedIndexWriter
            An instance of InvertedIndexWriter object into which each merged 
            postings list is written out one at a time
        """
        ### Begin your code
        temp_term = None
        temp_list = None
        for term, postings_list in heapq.merge(*indices):
            if term !=temp_term:
                if temp_term is None:
                    temp_term = term
                    temp_list = postings_list
                else:
                    merged_index.append(temp_term, list(sorted(set(temp_list))) )
                    temp_term = term
                    temp_list = postings_list
            else:
                temp_list = temp_list + postings_list
        if temp_term is not None:
            merged_index.append(temp_term, list(sorted(set(temp_list))))
        ### End your code

首先确保它在测试数据上正常运行

In [26]:
BSBI_instance = BSBIIndex(data_dir=toy_dir, output_dir = 'toy_output_dir', )
BSBI_instance.index()

接下来对整个数据集构建索引 (2%)

In [27]:
BSBI_instance = BSBIIndex(data_dir='pa1-data/pa1-data/', output_dir = 'output_dir', )
BSBI_instance.index()

如果你在合并阶段出现了错误，可以利用以下代码来debug。

In [28]:
BSBI_instance = BSBIIndex(data_dir='pa1-data/pa1-data/', output_dir = 'output_dir', )
BSBI_instance.intermediate_indices = ['index_'+str(i) for i in range(10)]
with InvertedIndexWriter(BSBI_instance.index_name, directory=BSBI_instance.output_dir, postings_encoding=BSBI_instance.postings_encoding) as merged_index:
    with contextlib.ExitStack() as stack:
        indices = [stack.enter_context(InvertedIndexIterator(index_id, directory=BSBI_instance.output_dir, postings_encoding=BSBI_instance.postings_encoding)) for index_id in BSBI_instance.intermediate_indices]
        BSBI_instance.merge(indices, merged_index)

<div style="background:#f7fafc; border-left:4px solid #4299e1; padding:10px 14px; border-radius:4px;">
到这里，我们已经完成了BSBI倒排索引最基础的构建，生成的索引已经可以支持基于词条的文档查询操作。接下来，将以此为基础进行功能与性能上的进一步优化。
</div>

## 布尔联合检索 (10%)

我们将实现BSBIIndex中的`retrieve`函数，对于给定的包含由空格分隔tokens的字符串查询，返回包含查询中所有tokens的文档列表。但是我们并不能迭代遍历整个索引或者将整个索引加载到内存来寻找相应的terms（索引太大）。

首先我们要实现`InvertedIndex`的子类`InvertedIndexMapper`，它能够找到对应terms在索引文件中位置并取出它的倒排记录表。 (2%)

<div style="background:#f7fafc; border-left:4px solid #4299e1; padding:10px 14px; border-radius:4px;">
当使用多个关键词进行查询时，应获取同时包含所有关键词的文档。为了更加高效地实现这一功能，我们要完成布尔联合检索，即多关键词的“与”查询。<br><br>
在本部分，需要实现_get_postings_list方法，主要依靠前期构建的索引结构完成。先通过传入的term从postings_dict中取出对应的存储位置、词条数量、字节长度三元组信息，接着将索引文件指针定位到该三元组的起始位置，读取指定字节长度的编码字节流，再对读取到的字节流进行解码得到倒排列表，最后将解码后的倒排列表直接返回即可。
</div>

In [29]:
class InvertedIndexMapper(InvertedIndex):
    def __getitem__(self, key):
        return self._get_postings_list(key)
    
    def _get_postings_list(self, term):
        """Gets a postings list (of docIds) for `term`.
        
        This function should not iterate through the index file.
        I.e., it should only have to read the bytes from the index file
        corresponding to the postings list for the requested term.
        """
        ### Begin your code
        if term not in self.postings_dict:
            return []
        
        start_position, number_of_posting, length_of_bytes = self.postings_dict[term]
        self.index_file.seek(start_position)
        postings_list_encoded = self.index_file.read(length_of_bytes)
        postings_list = self.postings_encoding.decode(postings_list_encoded)
        return postings_list
        ### End your code

写一些测试样例检测`_get_postings_list`的实现 (1%)

In [30]:
### Begin your code
with InvertedIndexWriter(index_name='test_map', directory='tmp/') as w:
    w.append(0, [1,2,3])

with InvertedIndexMapper(index_name='test_map', directory='tmp/') as m:
    assert m[0] == [1,2,3]
    assert m[999] == []
### End your code

现在我们获得了查询中terms对应的倒排记录表，接着需要求他们的交集。这部分与我们之前作业的merge方法类似，请实现`sorted_intersect`函数，遍历两个有序列表并在线性时间内合并。 (2%)

<div style="background:#f7fafc; border-left:4px solid #4299e1; padding:10px 14px; border-radius:4px;">
sorted_intersect函数的主要功能为求两个列表的交集，此处采用双指针算法实现。<br><br>
初始化指针p1、p2指向起始位置，创建result_list列表存储交集结果，然后开始循环。两个指针相互对比元素：由于已经是排好序的列表，较小值对应指针直接进行后移；若值相等，则将元素加入结果列表，两个指针同步后移。循环结束后直接返回结果列表，即为所求的交集。
</div>

In [31]:
def sorted_intersect(list1, list2):
    """Intersects two (ascending) sorted lists and returns the sorted result
    
    Parameters
    ----------
    list1: List[Comparable]
    list2: List[Comparable]
        Sorted lists to be intersected
        
    Returns
    -------
    List[Comparable]
        Sorted intersection        
    """
    ### Begin your code
    p1 = 0
    p2 = 0
    result_list = []
    while p1 < len(list1) and p2 < len(list2):
        if list1[p1] < list2[p2]:
            p1 += 1
        elif list1[p1] > list2[p2]:
            p2 += 1
        else:
            result_list.append(list1[p1])
            p1 += 1
            p2 += 1
    return result_list
    ### End your code

简单的测试样例 (1%)

In [32]:
### Begin your code
assert sorted_intersect([1,3,5], [3,5,7]) == [3,5]
assert sorted_intersect([1,2], [3,4]) == []
### End your code

现在可以利用`sorted_intersect` 和 `InvertedIndexMapper`来实现`retrieve`函数。 (3%)

<div style="background:#f7fafc; border-left:4px solid #4299e1; padding:10px 14px; border-radius:4px;">
retrieve 方法实现了布尔查询的主要功能，方法执行时先加载词条与文档的ID映射，通过InvertedIndexMapper高效读取磁盘上的倒排索引数据，先将输入的查询字符串按空格切分为独立的查询词，遍历所有查询词，若存在任意一个查询词未出现在数据集中，就直接返回空列表，对存在的查询词先转换为对应的term_id，再通过mapper获取该词条的有序倒排列表，依次对所有倒排列表执行有序交集运算，逐步筛选出同时包含所有查询词的doc_id集合，将匹配到的doc_id通过文档ID 映射转换为原始文档路径名称，最终返回排序后的文档名称列表，完成精准的联合查询检索。
</div>

In [33]:
# %%tee submission/retrieve.py
class BSBIIndex(BSBIIndex):
    def retrieve(self, query):
        """Retrieves the documents corresponding to the conjunctive query
        
        Parameters
        ----------
        query: str
            Space separated list of query tokens
            
        Result
        ------
        List[str]
            Sorted list of documents which contains each of the query tokens. 
            Should be empty if no documents are found.
        
        Should NOT throw errors for terms not in corpus
        """
        if len(self.term_id_map) == 0 or len(self.doc_id_map) == 0:
            self.load()

        ### Begin your code
        with InvertedIndexMapper(self.index_name, directory=self.output_dir, 
                                 postings_encoding=
                                 self.postings_encoding) as mapper:
            result = None
            term_list = query.split()
            for term in term_list:
                term_id = self.term_id_map.str_to_id.get(term)
                if term_id is None:
                    return []
                else:
                    temp_list = mapper._get_postings_list(term_id)
                    if result is None:
                        result = temp_list
                    else:
                        result = sorted_intersect(result, temp_list)
            result_str_list = []
            for i in result:
                temp_str = self.doc_id_map._get_str(i)
                result_str_list.append(temp_str)
            return result_str_list
        ### End your code

通过一个简单的查询来测试索引在真实数据集上是否正常工作。

In [34]:
BSBI_instance = BSBIIndex(data_dir='pa1-data/pa1-data', output_dir = 'output_dir', )
BSBI_instance.retrieve('boolean retrieval')

['1\\cs276.stanford.edu_',
 '1\\cs276a.stanford.edu_',
 '3\\infolab.stanford.edu_TR_CS-TN-95-21.html',
 '4\\nlp.stanford.edu_IR-book_',
 '4\\nlp.stanford.edu_IR-book_html_htmledition_an-appraisal-of-probabilistic-models-1.html',
 '4\\nlp.stanford.edu_IR-book_html_htmledition_bayesian-network-approaches-to-ir-1.html',
 '4\\nlp.stanford.edu_IR-book_html_htmledition_boolean-retrieval-2.html',
 '4\\nlp.stanford.edu_IR-book_html_htmledition_computing-scores-in-a-complete-search-system-1.html',
 '4\\nlp.stanford.edu_IR-book_html_htmledition_dictionaries-and-tolerant-retrieval-1.html',
 '4\\nlp.stanford.edu_IR-book_html_htmledition_efficient-scoring-and-ranking-1.html',
 '4\\nlp.stanford.edu_IR-book_html_htmledition_inexact-top-k-document-retrieval-1.html',
 '4\\nlp.stanford.edu_IR-book_html_htmledition_probabilistic-information-retrieval-1.html',
 '4\\nlp.stanford.edu_IR-book_html_htmledition_putting-it-all-together-1.html',
 '4\\nlp.stanford.edu_IR-book_html_htmledition_references-and-furth

你也可以通过读取文件来测试其中的页面是否真的包含了查询的terms

In [35]:
with open("pa1-data/pa1-data/1/cs276.stanford.edu_", 'r') as f:
    print(f.read())

cs276 information retrieval and web search cs 276 ling 286 information retrieval and web search spring 2011 pandu nayak and prabhakar raghavan lecture 3 units tu th 4 15 5 30 gates b03 tas sonali aggarwal ivan cui valentin spitkovsky and sandeep sripada staff e mail cs276 spr1011 staff lists stanford edu lectures are also available online and on television through scpd sitn course description basic and advanced techniques for text based information systems efficient text indexing boolean and vector space retrieval models evaluation and interface issues web search including crawling link based algorithms and web metadata text web clustering classification text mining syllabus additional information staff contact information piazzza we strongly recommend students to post questions to the course page on www piazzza com instead of sending emails this forum enables students to discuss the questions they encounter in lectures or assignments here is a quick introduction video email if you hav

测试dev queries（提前构建好的查询与结果）是否正确 (1%)

In [36]:
for i in range(1, 9):
    with open('dev_queries/dev_queries/query.' + str(i)) as q:
        query = q.read()
        my_results = [os.path.normpath(path) for path in BSBI_instance.retrieve(query)]
        with open('dev_output/dev_output/' + str(i) + '.out') as o:
            reference_results = [os.path.normpath(x.strip()) for x in o.readlines()]
            assert my_results == reference_results, "Results DO NOT match for query: "+query.strip()
        print("Results match for query:", query.strip())

Results match for query: we are
Results match for query: stanford class
Results match for query: stanford students
Results match for query: very cool
Results match for query: the
Results match for query: a
Results match for query: the the
Results match for query: stanford computer science


如果出现了错误，可以通过以下代码比较输出与正确结果的差异

In [37]:
set(my_results) - set(reference_results)

set()

In [38]:
set(reference_results) - set(my_results)

set()

确保你构建自己的查询来测试所有可能的边界情况，例如数据集中没有出现的terms，或者拖慢合并的频繁出现的terms。

<div style="background:#f7fafc; border-left:4px solid #4299e1; padding:10px 14px; border-radius:4px;">
经过上述代码，成功实现了布尔联合检索的功能。整体来看，BSBI在功能上已经趋于完善，接下来将对编码进行优化，使用更为高效的编码方式来减少倒排表的内存占用。
</div>

# 索引压缩  (30%)

在这部分，你将使用可变长字节编码对索引进行压缩。（gap-encoding可选，对gap进行编码）

下面几个Python运算符可能对你有帮助

In [51]:
# Remainder (modulo) operator %

print("10 % 2 = ", 10 % 2)
print("10 % 3 = ", 10 % 3)

# Integer division in Python 3 is done by using two slash signs

print("10 / 3 = ", 10 / 3)
print("10 // 3 = ", 10 // 3)

10 % 2 =  0
10 % 3 =  1
10 / 3 =  3.3333333333333335
10 // 3 =  3


完成下面的`CompressedPostings`类，我们将用它来替换`UncompressedPostings`。关于如何使用gap-encoding和可变长字节编码，你可以参考老师课件或者[Chapter 5](https://nlp.stanford.edu/IR-book/pdf/05comp.pdf)。 (18%)

<div style="background:#f7fafc; border-left:4px solid #4299e1; padding:10px 14px; border-radius:4px;">
这里的CompressedPostings类使用间隙编码(Gap-Encoding)与可变字节编码(VB-Encoding)相结合的方式进行优化，核心是通过压缩算法大幅减少倒排列表的磁盘存储空间，提升索引读写效率。主要实现分为倒排列表压缩与解压缩这两部分。<br><br>
类中先实现了两个静态辅助方法完成单个数值的可变字节编解码，其中_vb_encode方法将数值拆分为128进制的字节序列，通过为最后一个字节添加最高位标记来标识编码结束，_vb_decode方法则逐字节解析字节流，根据结束标记还原出原始的数值；在encode方法中，先对有序的文档ID列表处理得到间隙列表（用后一个文档ID减去前一个文档ID），再对每一个间隙值调用可变字节编码方法，将所有编码后的字节流拼接，最终返回压缩后的字节数据；在decode解压缩方法中，先逐段读取带有结束标记的可变字节单元，调用解码方法还原出间隙值，再通过累加间隙值的方式还原出原始的有序文档ID列表，最终实现完整的解压缩，还原出原来的倒排文档ID列表。
</div>

In [52]:
class CompressedPostings:
    #If you need any extra helper methods you can add them here 
    ### Begin your code
    @staticmethod
    def _vb_encode(num):
        bytes_list = []
        while True:
            bytes_list.insert(0, num % 128)
            if num < 128:
                break
            num = num // 128
        bytes_list[-1] += 128
        return bytes(bytes_list)
    
    @staticmethod
    def _vb_decode(byte_stream):
        num = 0
        for byte in byte_stream:
            if byte < 128:
                num = num * 128 + byte
            else:
                num = num * 128 + (byte - 128)
                return num
        return num
    ### End your code
    
    @staticmethod
    def encode(postings_list):
        """Encodes `postings_list` using gap encoding with variable byte 
        encoding for each gap
        
        Parameters
        ----------
        postings_list: List[int]
            The postings list to be encoded
        
        Returns
        -------
        bytes: 
            Bytes reprsentation of the compressed postings list 
            (as produced by `array.tobytes` function)
        """
        ### Begin your code
        if not postings_list:
            return b''
        gaps = []
        prev = 0
        for did in sorted(postings_list):
            gaps.append(did - prev)
            prev = did
        encoded = b''
        for g in gaps:
            encoded += CompressedPostings._vb_encode(g)
        return encoded
        ### End your code

        
    @staticmethod
    def decode(encoded_postings_list):
        """Decodes a byte representation of compressed postings list
        
        Parameters
        ----------
        encoded_postings_list: bytes
            Bytes representation as produced by `CompressedPostings.encode` 
            
        Returns
        -------
        List[int]
            Decoded postings list (each posting is a docIds)
        """
        ### Begin your code
        if not encoded_postings_list:
            return []
        postings = []
        idx = 0
        prev = 0
        while idx < len(encoded_postings_list):
            byte_arr = []
            while idx < len(encoded_postings_list) and encoded_postings_list[idx] < 128:
                byte_arr.append(encoded_postings_list[idx])
                idx += 1
            byte_arr.append(encoded_postings_list[idx])
            idx += 1
            gap = CompressedPostings._vb_decode(byte_arr)
            prev += gap
            postings.append(prev)
        return postings
        ### End your code


首先，如果你实现了额外的辅助函数，写一些测试样例

In [53]:
### Begin your code

### End your code

我们实现了一个简单的函数来测试你编码的列表是否被正确解码

In [54]:
def test_encode_decode(l):
    e = CompressedPostings.encode(l)
    d = CompressedPostings.decode(e)
    assert d == l
    print(l, e)

写一些测试样例来确保文档列表的压缩和解压正常运行 (2%)

In [55]:
### Begin your code
test_encode_decode([1,3,5,7,9])
test_encode_decode([10,20,30])
### End your code

[1, 3, 5, 7, 9] b'\x81\x82\x82\x82\x82'
[10, 20, 30] b'\x8a\x8a\x8a'


现在让我们创建一个新的输出文件夹并使用`CompressedPostings`来构建压缩索引

In [56]:
try: 
    os.mkdir('output_dir_compressed')
except FileExistsError:
    pass

In [ ]:
BSBI_instance_compressed = BSBIIndex(data_dir='pa1-data/pa1-data', output_dir = 'output_dir_compressed', postings_encoding=CompressedPostings)
BSBI_instance_compressed.index()

In [ ]:
BSBI_instance_compressed = BSBIIndex(data_dir='pa1-data/pa1-data', output_dir = 'output_dir_compressed', postings_encoding=CompressedPostings)
BSBI_instance_compressed.retrieve('boolean retrieval')

['1\\cs276.stanford.edu_',
 '1\\cs276a.stanford.edu_',
 '3\\infolab.stanford.edu_TR_CS-TN-95-21.html',
 '4\\nlp.stanford.edu_IR-book_',
 '4\\nlp.stanford.edu_IR-book_html_htmledition_an-appraisal-of-probabilistic-models-1.html',
 '4\\nlp.stanford.edu_IR-book_html_htmledition_bayesian-network-approaches-to-ir-1.html',
 '4\\nlp.stanford.edu_IR-book_html_htmledition_boolean-retrieval-2.html',
 '4\\nlp.stanford.edu_IR-book_html_htmledition_computing-scores-in-a-complete-search-system-1.html',
 '4\\nlp.stanford.edu_IR-book_html_htmledition_dictionaries-and-tolerant-retrieval-1.html',
 '4\\nlp.stanford.edu_IR-book_html_htmledition_efficient-scoring-and-ranking-1.html',
 '4\\nlp.stanford.edu_IR-book_html_htmledition_inexact-top-k-document-retrieval-1.html',
 '4\\nlp.stanford.edu_IR-book_html_htmledition_probabilistic-information-retrieval-1.html',
 '4\\nlp.stanford.edu_IR-book_html_htmledition_putting-it-all-together-1.html',
 '4\\nlp.stanford.edu_IR-book_html_htmledition_references-and-furth

像之前一样，用已构造好的查询语句来测试是否正常运行 (5%)

In [ ]:
for i in range(1, 9):
    with open('dev_queries/dev_queries/query.' + str(i)) as q:
        query = q.read()
        my_results = [os.path.normpath(path) for path in BSBI_instance_compressed.retrieve(query)]
        with open('dev_output/dev_output/' + str(i) + '.out') as o:
            reference_results = [os.path.normpath(x.strip()) for x in o.readlines()]
            assert my_results == reference_results, "Results DO NOT match for query: "+query.strip()
        print("Results match for query:", query.strip())

Results match for query: we are
Results match for query: stanford class
Results match for query: stanford students
Results match for query: very cool
Results match for query: the
Results match for query: a
Results match for query: the the
Results match for query: stanford computer science


请追加压缩前后的文件大小截图 (5%)

![这是图片](./.image/unc.jpg "UNC")
    未压缩
![这是图片](./.image/vbc.jpg "VBC")
    压缩后
<br>
<div style="background:#f7fafc; border-left:4px solid #4299e1; padding:10px 14px; border-radius:4px;">
如图所示，文件大小大幅下降，压缩率41.52%
</div>

# 额外的编码方式 (10%)

通过补充`ECCompressedPostings`的`encode` 和 `decode`方法来实现一种额外的索引压缩方式。在我们课上学的就是**gamma-encoding** 。另外如果大家感兴趣的话也可以了解**Delta Encoding** ，[ALGORITHM SIMPLE-9](https://github.com/manning/CompressionAlgorithms#simple-9) 等。

你应该以多字节（而不是bits）来存储倒排记录表，因为索引的长度和位置都存的是字节信息。 (4%)

<div style="background:#f7fafc; border-left:4px solid #4299e1; padding:10px 14px; border-radius:4px;">
在额外的编码方式这部分，本实验中采用Gap-Encoding和Gamma—Encoding实现，代码先通过静态方法GammaEncode完成单个正整数的伽马编码，将数字转为标准的伽马二进制格式；编码时先对有序文档 ID 计算间隙值，通过+1来适配伽马编码的正整数要求，再批量编码转换为二进制串，通过补零对齐后转为字节流进行输出；解码时则是将字节流还原为二进制串，去除填充位后，逐段解析伽马编码还原为间隙值，再通过累加间隙恢复原始文档 ID 列表。
</div>

In [48]:
class ECCompressedPostings:
    #If you need any extra helper methods you can add them here 
    ### Begin your code
    @staticmethod
    def GammaEncode(num):
        bin_str = bin(num)[2:]
        return '1' * (len(bin_str) - 1) + '0' + bin_str[1:]
    ### End your code
    
    @staticmethod
    def encode(postings_list):
        """Encodes `postings_list` 
        
        Parameters
        ----------
        postings_list: List[int]
            The postings list to be encoded
        
        Returns
        -------
        bytes: 
            Bytes reprsentation of the compressed postings list 
        """
        ### Begin your code
        if not postings_list:
            return b''
        
        gaps = []
        prev = 0
        for did in postings_list:
            gap = did - prev
            gaps.append(gap + 1)  # +1保证数值≥1
            prev = did
        
        encoded_bits = ''.join([ECCompressedPostings.GammaEncode(gap) for gap in gaps])
        pad_length = (8 - len(encoded_bits) % 8) % 8
        encoded_bits = '0' * pad_length + encoded_bits
        
        byte_array = bytearray()
        for i in range(0, len(encoded_bits), 8):
            byte = int(encoded_bits[i:i+8], 2)
            byte_array.append(byte)
        
        return array.array('B', byte_array).tobytes()
        ### End your code

        
    @staticmethod
    def decode(encoded_postings_list):
        """Decodes a byte representation of compressed postings list
        
        Parameters
        ----------
        encoded_postings_list: bytes
            Bytes representation as produced by `CompressedPostings.encode` 
            
        Returns
        -------
        List[int]
            Decoded postings list (each posting is a docId)
        """
        ### Begin your code
        if not encoded_postings_list:
            return []
        
        byte_array = array.array('B')
        byte_array.frombytes(encoded_postings_list)
        bit_str = ''.join([bin(byte)[2:].zfill(8) for byte in byte_array])
        
        bit_str = bit_str.lstrip('0')
        ptr = 0
        gap_list = []
        
        while ptr < len(bit_str):
            length = 0
            while ptr < len(bit_str) and bit_str[ptr] == '1':
                length += 1
                ptr += 1
            ptr += 1
            payload = bit_str[ptr:ptr+length]
            gap = int('1' + payload, 2) if length > 0 else 1
            gap_list.append(gap)
            ptr += length
        
        doc_ids = []
        prev = 0
        for gap in gap_list:
            prev += (gap - 1)
            doc_ids.append(prev)
        
        return doc_ids

        ### End your code

同上，写一些测试样例来确保代码正常运行 (3%)

In [49]:
### Begin your code
lst = [2,5,9,12]
e = ECCompressedPostings.encode(lst)
d = ECCompressedPostings.decode(e)
print(lst)
print(e)
print(d)
### End your code

[2, 5, 9, 12]
b'\x02\xe38'
[2, 5, 9, 12]


In [50]:
try: 
    os.mkdir('output_dir_eccompressed')
except FileExistsError:
    pass

BSBI_instance_compressed = BSBIIndex(data_dir='pa1-data/pa1-data', output_dir = 'output_dir_eccompressed', postings_encoding=ECCompressedPostings)
BSBI_instance_compressed.index()

请追加压缩前后的文件大小截图 (3%)

![这是图片](./.image/egc.jpg "EGC")
<div style="background:#f7fafc; border-left:4px solid #4299e1; padding:10px 14px; border-radius:4px;">
如图所示，文件大小进一步下降，相对于未压缩的压缩率达到33.11%
</div>

# 作业提交

本次作业用时约3周，截止日期为2026.04.09。请大家在截止日期前将代码、实验报告（可单独撰写，也可整合在jupyter notebook中）一起提交到ir24fall@163.com，邮件和文件命名方式均为`学号_姓名_hw1`，如`1811412_xxx_hw1.ipynb`